## 🔧 Step 1: Install Mantis

We begin by installing the `mantis` package directly from GitHub. This will download the latest version of the code and install all required dependencies.

In [1]:
# Clone the full Mantis repository and move into it
!git clone https://github.com/carsondudley1/Mantis.git
%cd Mantis

# Install the package in editable mode
!pip install -e .

Cloning into 'Mantis'...
remote: Enumerating objects: 234, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 234 (delta 38), reused 2 (delta 2), pack-reused 155 (from 1)
Receiving objects: 100% (234/234), 218.61 KiB | 1.66 MiB/s, done.
Resolving deltas: 100% (120/120), done.
/content/Mantis
Obtaining file:///content/Mantis
  Preparing metadata (setup.py) ... done
  Running setup.py develop for mantis


## 📦 Step 2: Download the Pretrained Model

Mantis uses pretrained simulation-grounded foundation models for forecasting.  
In this tutorial, we’ll use the **4-week horizon model with covariates**.

This cell downloads the model file (~1 GB) from the [GitHub release](https://github.com/carsondudley1/Mantis/releases/tag/mantis-v1.0) and places it in a local `models/` directory.

In [2]:
# Download the 4-week model that takes covariates

!mkdir -p models
!wget -O models/mantis_4w_cov.pt https://github.com/carsondudley1/Mantis/releases/download/mantis-v1.0/mantis_4w_cov.pt

--2026-03-18 16:04:48--  https://github.com/carsondudley1/Mantis/releases/download/mantis-v1.0/mantis_4w_cov.pt
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/1028419690/2f95c870-6a12-4737-a6f9-a9332d52ae01?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-18T16%3A43%3A07Z&rscd=attachment%3B+filename%3Dmantis_4w_cov.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-18T15%3A42%3A33Z&ske=2026-03-18T16%3A43%3A07Z&sks=b&skv=2018-11-09&sig=1rClRU5v81ePNNG8HfOJiz33F8cSQne3GpboBmLrqp0%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3Mzg1MzQ4OCwibmJmIjoxNzczODQ5ODg4LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjd

## 📊 Step 3: Import Mantis and Load Data

We now import the `Mantis` model and load Ebola case and death data. This dataset includes daily case and death counts for Ethiopia.

In [7]:
# Import Mantis and supporting libraries
from mantis import Mantis
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from google.colab import drive
drive.mount('/content/drive')


# Load COVID-19 hospitalizations and deaths datasets

cholera_data = pd.read_csv('/content/drive/My Drive/MantisExperiments/Data/cholera_data_who.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 🤖 Step 4: Initialize Mantis and other models and generate forecasts with metrics

We now create an instance of the Mantis model — specifically, the 4-week horizon version with covariates, as well as our evaluation models.  
Then we pass in Ethiopia's case and death time series to generate forecasts.

Mantis returns 9 quantiles for each of the next 4 weeks, providing a full probabilistic forecast.


In [8]:
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from statsmodels.tsa.api import ETSModel
from statsmodels.tsa.statespace.sarimax import SARIMAX
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

# ── Config ──────────────────────────────────────────────────────────────────────
country = "AFR::ETH"
FORECAST_HORIZON = 4
STRIDE = 1
START_WEEK = 52
LOOKBACK = 16
LSTM_EPOCHS = 10
LSTM_HIDDEN = 64
LSTM_LAYERS = 2
LSTM_DROPOUT = 0.2
LSTM_BATCH = 32
LSTM_LR = 1e-3

QUANTILES = np.array([0.05, 0.1, 0.25, 0.4, 0.5, 0.6, 0.75, 0.9, 0.95])
MEDIAN_IDX = int(np.where(QUANTILES == 0.5)[0][0])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Data setup ──────────────────────────────────────────────────────────────────
country_df = cholera_data[cholera_data["Location"] == country].sort_values("reporting_date")
cases_ts = country_df["sCh"].fillna(0).values.astype(float)
deaths_ts = country_df["deaths"].fillna(0).values.astype(float)

# target = deaths, covariate = cases
y_full = deaths_ts
x_full = cases_ts

# ── Mantis model ────────────────────────────────────────────────────────────────
model = Mantis(forecast_horizon=FORECAST_HORIZON, use_covariate=True)

# ── Helper functions ────────────────────────────────────────────────────────────
def make_gaussian_quantiles(mean, stderr, quantiles):
    q = np.zeros((len(mean), len(quantiles)))
    for h in range(len(mean)):
        for j, qq in enumerate(quantiles):
            q[h, j] = mean[h] + stats.norm.ppf(qq) * stderr[h]
    return np.maximum(q, 0.0)

def ensure_monotonic(q_preds):
    out = q_preds.copy()
    for h in range(out.shape[0]):
        out[h, :] = np.sort(out[h, :])
    return out

def compute_window_metrics(true_future, q_preds, baseline_pred):
    median = q_preds[:, MEDIAN_IDX]
    model_abs = np.abs(true_future - median)
    base_abs = np.abs(true_future - baseline_pred)
    cov90_hits = np.sum((true_future >= q_preds[:, 0]) & (true_future <= q_preds[:, 8]))
    cov50_hits = np.sum((true_future >= q_preds[:, 2]) & (true_future <= q_preds[:, 6]))
    return model_abs, base_abs, int(cov90_hits), int(cov50_hits)

def dm_test(loss1, loss2, h=1):
    d = np.array(loss1) - np.array(loss2)
    if len(d) == 0:
        return np.nan
    mean_d = np.mean(d)
    var_d = np.var(d, ddof=0)
    for k in range(1, h):
        cov_k = np.mean((d[:-k] - mean_d) * (d[k:] - mean_d))
        var_d += 2 * cov_k
    if var_d <= 1e-8:
        return np.nan
    dm_stat = mean_d / np.sqrt(var_d / len(d))
    return 2 * (1 - stats.norm.cdf(abs(dm_stat)))

# ── Forecasting functions ───────────────────────────────────────────────────────
def forecast_mantis(y_hist, x_hist):
    pred = model.predict(
        time_series=y_hist,
        covariate=x_hist,
        target_type=2,
        covariate_type=0,
    )
    return ensure_monotonic(pred)

def forecast_ets(y_hist, x_hist, x_future):
    x_design = np.column_stack([np.ones(len(x_hist)), x_hist])
    beta, *_ = np.linalg.lstsq(x_design, y_hist, rcond=None)
    a, b = float(beta[0]), float(beta[1])
    resid_hist = y_hist - (a + b * x_hist)
    resid_series = pd.Series(resid_hist.astype(float))

    use_seasonal = len(resid_hist) >= 2 * 52 and np.nanstd(resid_hist) > 1e-8
    specs = []
    if use_seasonal:
        specs.append(dict(error="add", trend="add", seasonal="add", seasonal_periods=52))
    specs += [
        dict(error="add", trend="add", seasonal=None),
        dict(error="add", trend=None, seasonal=None),
    ]
    for spec in specs:
        try:
            fit = ETSModel(resid_series, **spec).fit(disp=False)
            pred_obj = fit.get_prediction(
                start=len(resid_hist),
                end=len(resid_hist) + FORECAST_HORIZON - 1,
            )
            resid_mean = np.asarray(pred_obj.predicted_mean, dtype=float)
            in_resid = np.asarray(fit.resid, dtype=float)
            in_resid = in_resid[np.isfinite(in_resid)]
            resid_std = max(np.nanstd(in_resid, ddof=1), 1e-3)
            horizon_se = resid_std * np.sqrt(np.arange(1, FORECAST_HORIZON + 1))
            total_mean = (a + b * x_future) + resid_mean
            return ensure_monotonic(make_gaussian_quantiles(total_mean, horizon_se + 1e-6, QUANTILES))
        except Exception:
            continue
    raise RuntimeError("All ETS specs failed")

def forecast_sarimax(y_hist, x_hist, x_future):
    exog_hist = x_hist.reshape(-1, 1)
    exog_future = x_future.reshape(-1, 1)
    seasonal_order = (1, 0, 0, 52) if len(y_hist) >= 3 * 52 else (0, 0, 0, 0)
    fit = SARIMAX(
        y_hist, exog=exog_hist,
        order=(1, 1, 1), seasonal_order=seasonal_order,
        trend="n", simple_differencing=True,
        enforce_stationarity=False, enforce_invertibility=False,
    ).fit(disp=False, method="lbfgs", maxiter=40)
    fc = fit.get_forecast(steps=FORECAST_HORIZON, exog=exog_future)
    mean = np.asarray(fc.predicted_mean, dtype=float)
    se = np.asarray(fc.se_mean, dtype=float) if hasattr(fc, "se_mean") and fc.se_mean is not None \
        else np.sqrt(np.maximum(np.asarray(fc.var_pred_mean, dtype=float), 1e-8))
    return ensure_monotonic(make_gaussian_quantiles(mean, se + 1e-6, QUANTILES))

# ── LSTM model definition ──────────────────────────────────────────────────────
class QuantileLoss(nn.Module):
    def __init__(self, quantiles):
        super().__init__()
        self.quantiles = torch.tensor(quantiles, dtype=torch.float32)
    def forward(self, preds, target):
        target_exp = target.unsqueeze(-1).expand_as(preds)
        q = self.quantiles.to(preds.device).view(1, 1, -1)
        errors = target_exp - preds
        return torch.max((q - 1) * errors, q * errors).mean()

class LSTMQuantileForecaster(nn.Module):
    def __init__(self, hidden_dim, horizon, n_quantiles, num_layers=2, dropout=0.2):
        super().__init__()
        self.horizon = horizon
        self.n_quantiles = n_quantiles
        self.lstm = nn.LSTM(input_size=2, hidden_size=hidden_dim,
                            num_layers=num_layers,
                            dropout=dropout if num_layers > 1 else 0.0,
                            batch_first=True)
        self.bn = nn.BatchNorm1d(hidden_dim)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim + horizon, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, horizon * n_quantiles),
        )
    def forward(self, x_hist, x_future_cov, use_bn=True):
        out, _ = self.lstm(x_hist)
        h = out[:, -1, :]
        if use_bn and h.shape[0] > 1:
            h = self.bn(h)
        z = torch.cat([h, x_future_cov], dim=1)
        return self.head(z).view(-1, self.horizon, self.n_quantiles)

def forecast_lstm(y_hist, x_hist, x_future):
    n = len(y_hist)
    if n < LOOKBACK + FORECAST_HORIZON + 1:
        raise ValueError("Not enough history for LSTM")
    # Build training windows from history
    x_hist_all, x_fc_all, y_all = [], [], []
    for i in range(LOOKBACK, n - FORECAST_HORIZON + 1):
        x_hist_all.append(np.column_stack([y_hist[i-LOOKBACK:i], x_hist[i-LOOKBACK:i]]))
        x_fc_all.append(x_hist[i:i+FORECAST_HORIZON])
        y_all.append(y_hist[i:i+FORECAST_HORIZON])
    x_hist_all = np.asarray(x_hist_all, dtype=float)
    x_fc_all = np.asarray(x_fc_all, dtype=float)
    y_all = np.asarray(y_all, dtype=float)
    if len(x_hist_all) < 2:
        raise ValueError("Too few samples for LSTM")

    sc_h = StandardScaler()
    sc_f = StandardScaler()
    sc_y = StandardScaler()
    x_hist_s = sc_h.fit_transform(x_hist_all.reshape(-1, 2)).reshape(x_hist_all.shape)
    x_fc_s = sc_f.fit_transform(x_fc_all)
    y_s = sc_y.fit_transform(y_all.reshape(-1, 1)).reshape(y_all.shape)

    ds = TensorDataset(
        torch.tensor(x_hist_s, dtype=torch.float32, device=device),
        torch.tensor(x_fc_s, dtype=torch.float32, device=device),
        torch.tensor(y_s, dtype=torch.float32, device=device),
    )
    dl = DataLoader(ds, batch_size=min(LSTM_BATCH, len(ds)), shuffle=True, drop_last=False)

    net = LSTMQuantileForecaster(LSTM_HIDDEN, FORECAST_HORIZON, len(QUANTILES),
                                  LSTM_LAYERS, LSTM_DROPOUT).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=LSTM_LR)
    loss_fn = QuantileLoss(QUANTILES)
    net.train()
    for _ in range(LSTM_EPOCHS):
        for xb_h, xb_f, yb in dl:
            opt.zero_grad()
            loss_fn(net(xb_h, xb_f, use_bn=True), yb).backward()
            opt.step()

    # Inference
    hist_now = np.column_stack([y_hist[-LOOKBACK:], x_hist[-LOOKBACK:]])
    fut_now = x_future.reshape(1, -1)
    hist_t = torch.tensor(sc_h.transform(hist_now).reshape(1, LOOKBACK, 2),
                          dtype=torch.float32, device=device)
    fut_t = torch.tensor(sc_f.transform(fut_now), dtype=torch.float32, device=device)
    net.eval()
    with torch.no_grad():
        q_scaled = net(hist_t, fut_t, use_bn=False).squeeze(0).cpu().numpy()
    q = np.zeros_like(q_scaled)
    for qi in range(q.shape[1]):
        q[:, qi] = sc_y.inverse_transform(q_scaled[:, qi].reshape(-1, 1)).ravel()
    return ensure_monotonic(np.maximum(q, 0.0))

# ── Rolling evaluation loop ─────────────────────────────────────────────────────
methods = ["mantis", "ets_x", "sarimax_x", "lstm_q"]
stats_store = {
    m: {"model_abs": [], "baseline_abs": [],
        "cov90_hits": 0, "cov50_hits": 0, "total_points": 0, "fallbacks": 0}
    for m in methods
}

# Also keep per-window forecasts for later plotting
all_forecasts = {m: [] for m in methods}
all_ground_truths = []
all_start_weeks = []

forecast_weeks = list(range(START_WEEK, len(y_full) - FORECAST_HORIZON + 1, STRIDE))
print(f"Rolling evaluation: {len(forecast_weeks)} windows, horizon={FORECAST_HORIZON}, stride={STRIDE}")

for t in tqdm(forecast_weeks, desc="Eval windows"):
    y_hist = y_full[:t]
    x_hist = x_full[:t]
    y_true = y_full[t:t + FORECAST_HORIZON]
    if len(y_true) < FORECAST_HORIZON:
        continue

    baseline_pred = np.repeat(y_hist[-1], FORECAST_HORIZON)
    x_future = np.repeat(x_hist[-1], FORECAST_HORIZON)

    all_ground_truths.append(y_true)
    all_start_weeks.append(t)

    forecasters = {
        "mantis":    lambda: forecast_mantis(y_hist, x_hist),
        "ets_x":     lambda: forecast_ets(y_hist, x_hist, x_future),
        "sarimax_x": lambda: forecast_sarimax(y_hist, x_hist, x_future),
        "lstm_q":    lambda: forecast_lstm(y_hist, x_hist, x_future),
    }

    for mname, fn in forecasters.items():
        try:
            q_pred = fn()
        except Exception:
            stats_store[mname]["fallbacks"] += 1
            q_pred = np.tile(baseline_pred.reshape(-1, 1), (1, len(QUANTILES)))

        m_abs, b_abs, c90, c50 = compute_window_metrics(y_true, q_pred, baseline_pred)
        s = stats_store[mname]
        s["model_abs"].extend(m_abs)
        s["baseline_abs"].extend(b_abs)
        s["cov90_hits"] += c90
        s["cov50_hits"] += c50
        s["total_points"] += FORECAST_HORIZON
        all_forecasts[mname].append(q_pred)

# ── Summary table ───────────────────────────────────────────────────────────────
mantis_errors = stats_store["mantis"]["model_abs"]
rows = []
for mname in methods:
    s = stats_store[mname]
    model_mae = float(np.mean(s["model_abs"])) if s["model_abs"] else np.nan
    base_mae = float(np.mean(s["baseline_abs"])) if s["baseline_abs"] else np.nan
    rel_mae = model_mae / base_mae if (np.isfinite(base_mae) and base_mae != 0) else np.nan
    cov90 = s["cov90_hits"] / s["total_points"] if s["total_points"] > 0 else np.nan
    cov50 = s["cov50_hits"] / s["total_points"] if s["total_points"] > 0 else np.nan
    dm_p = np.nan if mname == "mantis" else dm_test(mantis_errors, s["model_abs"], h=FORECAST_HORIZON)
    rows.append({
        "method": mname, "model_mae": model_mae, "relative_mae": rel_mae,
        "coverage_50": cov50, "coverage_90": cov90,
        "dm_p_value": dm_p, "fallbacks": s["fallbacks"],
        "forecast_points": s["total_points"],
    })

summary_df = pd.DataFrame(rows).sort_values("relative_mae")

print("\n=== Multimodel Probabilistic Rolling Evaluation ===")
print(f"Country: {country} | Target: deaths | Covariate: cases")
print(f"Horizon: {FORECAST_HORIZON} | Stride: {STRIDE} | Windows: {len(forecast_weeks)}")
print("Relative MAE = model MAE / naive baseline MAE (lower is better; <1 beats baseline)")
print("dm_p_value = Diebold-Mariano test vs Mantis (H0: equal accuracy)\n")
print(summary_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

Rolling evaluation: 197 windows, horizon=4, stride=1


Eval windows:   0%|          | 0/197 [00:00<?, ?it/s]


=== Multimodel Probabilistic Rolling Evaluation ===
Country: AFR::ETH | Target: deaths | Covariate: cases
Horizon: 4 | Stride: 1 | Windows: 197
Relative MAE = model MAE / naive baseline MAE (lower is better; <1 beats baseline)
dm_p_value = Diebold-Mariano test vs Mantis (H0: equal accuracy)

   method  model_mae  relative_mae  coverage_50  coverage_90  dm_p_value  fallbacks  forecast_points
   mantis     3.9584        0.9937       0.4378       0.8223         NaN          0              788
    ets_x     4.4329        1.1128       0.5203       0.8363      0.0095          0              788
   lstm_q     4.9517        1.2431       0.0457       0.1117      0.0005          0              788
sarimax_x     6.1817        1.5518       0.3490       0.6802      0.0000          0              788
